# 02. Run trial (steps ①-⑭)

| | |
|---|---|
| Input | Day folder: `Top/`, `Front/` videos, `setup/` (NPZ, YOLO weights, settings) |
| Output | `result/raw_N.xlsx`, `video_N.mp4`, `setup_N.png`, `final_N.xlsx`, `config.yaml` (N = trial number) |
| GPU | Required (T4) |

Cell 0 asks for the day folder (`Top/`, `Front/`, `setup/`, `result/`). The last folder is remembered; press Enter to reuse it.
If `setup/experiment_settings.yaml` does not exist, a form opens: fill it in and click **Save**. Otherwise the current settings are shown with **Edit** buttons.
For each trial: run cell 1 (enter the trial number), cell 2 (test run), then cell 3 (full run).

In [ ]:
#@title 0. Setup: repository, Drive, day folder, day settings
REPO_URL = "https://github.com/dodokim777-boop/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
from main_code import settings
from main_code.common.notebook_ui import ask_day_folder, ask_trial, day_settings

DAY = ask_day_folder()
SETTINGS_FILE = day_settings(DAY, os.path.join(REPO_DIR, 'experiment_settings.yaml'))

In [ ]:
#@title 1. Trial number (run again for each trial)
import torch
TRIAL = ask_trial(DAY)
settings.configure(settings.day_settings_path(DAY), data_root=DAY, trial=TRIAL)
settings.print_summary('preprocessing')
settings.check('preprocessing')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available (very slow on CPU)')

## 2. Test run

Processes the first frames only to check videos, models, and calibration.
Outputs are `result/test_raw.xlsx`, `test_video.mp4`, and `test_setup.png`, replaced at every test run.
In `test_setup.png`, the red dots should sit on the floor marker corners.

In [ ]:
#@title 2. Test run
TEST_FRAMES = 300  #@param {type:"integer"}
import cv2, matplotlib.pyplot as plt
from main_code.preprocessing.run_preprocessing import run_preprocessing

df_test = run_preprocessing(max_frames=TEST_FRAMES)
if os.path.exists(settings.VERIFY_IMAGE_PATH):
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(cv2.imread(settings.VERIFY_IMAGE_PATH), cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.show()

## 3. Full run

Preprocessing (①-⑤) then postprocessing (⑥-⑭) of the whole video. To inspect postprocessing step by step, use `03_detailed_analysis`.

In [ ]:
#@title 3. Full run (raw_N, video_N, setup_N, final_N)
from main_code.preprocessing.run_preprocessing import run_preprocessing
from main_code.postprocessing import run_postprocessing as P

df_raw = run_preprocessing()
ctx = P.step_load()
for step in P.STEPS:
    ctx = step(ctx)
ctx = P.step_save(ctx)

In [ ]:
#@title 4. Result plot
from main_code.common.result_plot import plot_result
fig = plot_result(ctx.sheets, title=f'Trial {TRIAL}')